# 🚀 Tema 6: Despliegue y Evaluación de Dashboards
## Lenguaje de Ciencia de Datos II (4364) — CIBERTEC · Ciclo 4 · 2026

---

Este notebook es la guía de referencia para la parte aplicativa del Tema 6.  
Cubrimos el ciclo completo: **ingesta → análisis → visualización → diseño → despliegue**.

| # | Sección | Contenido |
|---|---------|-----------|
| 0 | Setup y carga desde PokeAPI | Ingesta con caché Parquet |
| 1 | Exploración y preparación | Limpieza y feature engineering para el dashboard |
| 2 | Visualización estratégica | 5 gráficos con Plotly Express, cada uno justificado |
| 3 | Arquitectura de la Data App | Estructura modular del app.py |
| 4 | Patrones de caché | `@st.cache_data` y `@st.cache_resource` |
| 5 | Guía de despliegue | Streamlit Cloud y Render paso a paso |

**Dataset:** PokeAPI — primeros 150 Pokémon (Generación I)  
**Librerías:** `requests`, `pandas`, `plotly`, `streamlit`, `pathlib`, `pyarrow`


---
## ⚙️ Sección 0: Setup y Carga de Datos desde PokeAPI

### 3.2.1 — Diseño centrado en el usuario: primero los datos

Antes de diseñar cualquier interfaz, necesitamos entender qué datos tenemos disponibles.  
La PokeAPI es nuestra fuente; guardamos los resultados en Parquet para no sobrecargar la API en cada ejecución.


In [ ]:
# Instalación de dependencias (ejecutar solo la primera vez)
# !pip install requests pandas plotly pyarrow streamlit

In [19]:
import time
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from typing import Optional

# ── Constantes del curso ──────────────────────────────────────────
BASE_URL: str = "https://pokeapi.co/api/v2"
TOTAL_POKEMON: int = 386
CACHE_PATH: Path = Path("data/pokemon_gen1.parquet")
REQUEST_TIMEOUT: int = 10
SLEEP_BETWEEN_REQUESTS: float = 0.1

TIPO_COLORES: dict = {
    "fire": "#FF6B35", "water": "#4FC3F7", "grass": "#66BB6A",
    "electric": "#FFEE58", "psychic": "#EC407A", "normal": "#BDBDBD",
    "poison": "#AB47BC", "rock": "#8D6E63", "ground": "#FFA726",
    "flying": "#29B6F6", "bug": "#8BC34A", "ghost": "#5C6BC0",
    "ice": "#80DEEA", "dragon": "#7E57C2", "fighting": "#EF5350",
    "steel": "#B0BEC5", "fairy": "#F48FB1", "dark": "#546E7A",
}

print(">>> Setup completado correctamente.")
print(f">>> Cache path: {CACHE_PATH}")

>>> Setup completado correctamente.
>>> Cache path: data\pokemon_gen1.parquet


### Función de ingesta con manejo de errores

Siguiendo las convenciones del curso: `timeout=10`, `.raise_for_status()`, `try/except RequestException`.


In [20]:
def obtener_pokemon(pokemon_id: int) -> Optional[dict]:
    """Obtiene datos de un Pokémon desde la PokeAPI.

    Parameters
    ----------
    pokemon_id : int
        ID del Pokémon (1-150 para Generación I).

    Returns
    -------
    Optional[dict]
        Diccionario con los datos del Pokémon o None si falla.
    """
    try:
        url = f"{BASE_URL}/pokemon/{pokemon_id}"
        response = requests.get(url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()

        stats = {s["stat"]["name"]: s["base_stat"] for s in data["stats"]}
        tipos = [t["type"]["name"] for t in data["types"]]

        return {
            "id": data["id"],
            "nombre": data["name"].capitalize(),
            "tipo_primario": tipos[0] if len(tipos) > 0 else None,
            "tipo_secundario": tipos[1] if len(tipos) > 1 else None,
            "hp": stats.get("hp", 0),
            "ataque": stats.get("attack", 0),
            "defensa": stats.get("defense", 0),
            "ataque_especial": stats.get("special-attack", 0),
            "defensa_especial": stats.get("special-defense", 0),
            "velocidad": stats.get("speed", 0),
            "altura_dm": data["height"],
            "peso_hg": data["weight"],
            "experiencia_base": data["base_experience"],
        }
    except requests.exceptions.RequestException as e:
        print(f"[ERROR] Pokémon {pokemon_id}: {e}")
        return None


def cargar_dataset() -> pd.DataFrame:
    """Carga el dataset desde caché Parquet o lo descarga de la PokeAPI.

    Returns
    -------
    pd.DataFrame
        DataFrame con los datos de los 150 Pokémon de Generación I.
    """
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

    if CACHE_PATH.exists():
        print(f">>> Cargando desde caché: {CACHE_PATH}")
        return pd.read_parquet(CACHE_PATH)

    print(f">>> Descargando {TOTAL_POKEMON} Pokémon desde PokeAPI...")
    registros = []
    for i in range(1, TOTAL_POKEMON + 1):
        datos = obtener_pokemon(i)
        if datos:
            registros.append(datos)
        time.sleep(SLEEP_BETWEEN_REQUESTS)
        if i % 30 == 0:
            print(f"    {i}/{TOTAL_POKEMON} descargados...")

    df = pd.DataFrame(registros)
    df.to_parquet(CACHE_PATH, index=False)
    print(f">>> Dataset guardado en {CACHE_PATH}")
    return df

In [21]:
# Cargar el dataset (descarga o caché)
df = cargar_dataset()
print(f"\n>>> Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head()

>>> Cargando desde caché: data\pokemon_gen1.parquet

>>> Dataset cargado: 386 filas × 13 columnas


,id,nombre,tipo_primario,tipo_secundario,hp,ataque,defensa,ataque_especial,defensa_especial,velocidad,altura_dm,peso_hg,experiencia_base
0,1,Bulbasaur,grass,poison,45,49,49,65,65,45,7,69,64
1,2,Ivysaur,grass,poison,60,62,63,80,80,60,10,130,142
2,3,Venusaur,grass,poison,80,82,83,100,100,80,20,1000,236
3,4,Charmander,fire,None,39,52,43,60,50,65,6,85,62
4,5,Charmeleon,fire,None,58,64,58,80,65,80,11,190,142


---
## 🔍 Sección 1: Exploración y Preparación para el Dashboard

### 3.2.1 — ¿Qué preguntas de negocio responde nuestro dashboard?

Antes de elegir gráficos, identificamos las **preguntas clave** del usuario:

1. ¿Cuál es el tipo de Pokémon más común?
2. ¿Cuáles tipos tienen mayor ataque promedio?
3. ¿Cómo se distribuye el HP entre tipos?
4. ¿Existe relación entre ataque y defensa?
5. ¿Cuáles son los Pokémon más "extremos" en cada stat?


In [22]:
# Resumen general del dataset
print("=" * 50)
print("RESUMEN DEL DATASET")
print("=" * 50)
print(f"Total Pokémon    : {df.shape[0]}")
print(f"Tipos únicos     : {df['tipo_primario'].nunique()}")
print(f"Con tipo secundario: {df['tipo_secundario'].notna().sum()}")
print(f"Tipo más común   : {df['tipo_primario'].value_counts().index[0].upper()}")
print(f"Mayor ataque prom: {df.groupby('tipo_primario')['ataque'].mean().idxmax().upper()}")
print(f"HP promedio      : {df['hp'].mean():.1f}")
print(f"Mayor peso (hg)  : {df['peso_hg'].max()} ({df.loc[df['peso_hg'].idxmax(), 'nombre']})")
print("=" * 50)

RESUMEN DEL DATASET
Total Pokémon    : 386
Tipos únicos     : 17
Con tipo secundario: 182
Tipo más común   : WATER
Mayor ataque prom: DRAGON
HP promedio      : 66.5
Mayor peso (hg)  : 9500 (Groudon)


In [23]:
# Feature engineering: stats totales y categorías
STATS_COLS = ["hp", "ataque", "defensa", "ataque_especial", "defensa_especial", "velocidad"]

df["stat_total"] = df[STATS_COLS].sum(axis=1)
df["altura_m"] = df["altura_dm"] / 10
df["peso_kg"] = df["peso_hg"] / 10

# Categoría por stat total
df["tier"] = pd.cut(
    df["stat_total"],
    bins=[0, 300, 400, 500, 700],
    labels=["Débil", "Normal", "Fuerte", "Legendario"],
)

print(">>> Feature engineering completado.")
print(df[["nombre", "tipo_primario", "stat_total", "tier"]].head(10).to_string(index=False))

>>> Feature engineering completado.
    nombre tipo_primario  stat_total       tier
 Bulbasaur         grass         318     Normal
   Ivysaur         grass         405     Fuerte
  Venusaur         grass         525 Legendario
Charmander          fire         309     Normal
Charmeleon          fire         405     Fuerte
 Charizard          fire         534 Legendario
  Squirtle         water         314     Normal
 Wartortle         water         405     Fuerte
 Blastoise         water         530 Legendario
  Caterpie           bug         195      Débil


In [24]:
# Verificar valores nulos (calidad de datos)
print("Valores nulos por columna:")
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.any() else ">>> Sin valores nulos ✓")

# Estadísticas descriptivas de los stats
print("\nEstadísticas de los stats base:")
df[STATS_COLS].describe().round(1)

Valores nulos por columna:
tipo_secundario    204
dtype: int64

Estadísticas de los stats base:


,hp,ataque,defensa,ataque_especial,defensa_especial,velocidad
count,386.0,386.0,386.0,386.0,386.0,386.0
mean,66.5,71.8,68.9,66.7,67.8,64.5
std,28.2,28.5,30.6,27.7,27.8,27.2
min,1.0,5.0,5.0,10.0,20.0,5.0
25%,50.0,50.0,48.2,45.0,50.0,45.0
50%,63.5,70.0,65.0,65.0,65.0,64.0
75%,80.0,90.0,85.0,85.0,83.0,85.0
max,255.0,160.0,230.0,154.0,230.0,160.0


---
## 📊 Sección 2: Visualización Estratégica

### 3.2.2 — Del objetivo al gráfico correcto

Cada gráfico responde a una pregunta específica. **Nunca elegimos un gráfico por estética.**


#### Gráfico 1 — Barras horizontales: ¿Cuántos Pokémon hay por tipo?

**Objetivo:** Comparar categorías  
**Decisión:** Barras horizontales (mejor lectura de etiquetas largas)  
**Librería:** Plotly Express — interactividad nativa con tooltips


In [29]:
conteo_tipo = (
    df["tipo_primario"]
    .value_counts()
    .reset_index()
    .rename(columns={"tipo_primario": "tipo", "count": "cantidad"})
)
conteo_tipo["color"] = conteo_tipo["tipo"].map(TIPO_COLORES).fillna("#90A4AE")

fig1 = px.bar(
    conteo_tipo,
    x="cantidad",
    y="tipo",
    orientation="h",
    color="tipo",
    color_discrete_map=TIPO_COLORES,
    title="Distribución de Pokémon por Tipo Primario (Gen I)",
    labels={"cantidad": "Número de Pokémon", "tipo": "Tipo"},
    text="cantidad",
)
fig1.update_traces(textposition="outside")
fig1.update_layout(
    showlegend=False,
    plot_bgcolor="white",
    yaxis={"categoryorder": "total ascending"},
    height=500,
)
fig1.show()

#### Gráfico 2 — Scatter: ¿Existe relación entre Ataque y Defensa?

**Objetivo:** Explorar correlación entre dos variables cuantitativas  
**Decisión:** Scatter plot — ideal para detectar patrones y clusters  
**Añadido:** Color por tipo, tamaño por HP, tooltip con nombre


In [30]:
fig2 = px.scatter(
    df,
    x="ataque",
    y="defensa",
    color="tipo_primario",
)
fig2.update_layout(
    plot_bgcolor="white",
    height=500,
)
fig2.show()

In [31]:
display(df.head(2))

,id,nombre,tipo_primario,tipo_secundario,hp,ataque,defensa,ataque_especial,defensa_especial,velocidad,altura_dm,peso_hg,experiencia_base,stat_total,altura_m,peso_kg,tier
0,1,Bulbasaur,grass,poison,45,49,49,65,65,45,7,69,64,318,0.7,6.9,Normal
1,2,Ivysaur,grass,poison,60,62,63,80,80,60,10,130,142,405,1.0,13.0,Fuerte


In [9]:
fig2 = px.scatter(
    df,
    x="ataque",
    y="defensa",
    color="tipo_primario",
    color_discrete_map=TIPO_COLORES,
    size="hp",
    hover_name="nombre",
    hover_data={"tipo_primario": True, "stat_total": True, "hp": True},
    title="Ataque vs. Defensa — Coloreado por Tipo (tamaño = HP)",
    labels={"ataque": "Ataque Base", "defensa": "Defensa Base", "tipo_primario": "Tipo"},
)
fig2.update_layout(
    plot_bgcolor="white",
    height=500,
)
fig2.show()

#### Gráfico 3 — Box Plot: ¿Cómo se distribuye el HP por tipo?

**Objetivo:** Mostrar distribución estadística y outliers  
**Decisión:** Box plot — muestra mediana, IQR y valores extremos  
**Insight esperado:** Algunos tipos tienen HP muy variable (alta dispersión)


In [32]:
# Solo tipos con >= 3 Pokémon para que el box plot sea significativo
tipos_validos = df["tipo_primario"].value_counts()
tipos_validos = tipos_validos[tipos_validos >= 3].index

df_box = df[df["tipo_primario"].isin(tipos_validos)].copy()

fig3 = px.box(
    df_box,
    x="tipo_primario",
    y="hp",
    color="tipo_primario",
    color_discrete_map=TIPO_COLORES,
    hover_name="nombre",
    title="Distribución de HP por Tipo (tipos con ≥ 3 Pokémon)",
    labels={"hp": "HP Base", "tipo_primario": "Tipo"},
    points="all",
)
fig3.update_layout(
    showlegend=False,
    plot_bgcolor="white",
    height=450,
    xaxis_tickangle=-35,
)
fig3.show()

#### Gráfico 4 — Heatmap: ¿Qué tan correlacionados están los stats?

**Objetivo:** Mostrar correlaciones entre múltiples variables numéricas  
**Decisión:** Heatmap de correlación — estándar en análisis de datos  
**Insight clave:** ¿Pokémon rápidos tienden a tener menos defensa?


In [33]:
corr_matrix = df[STATS_COLS].corr().round(2)

fig4 = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns.tolist(),
    y=corr_matrix.columns.tolist(),
    colorscale="RdBu_r",
    zmid=0,
    text=corr_matrix.values.round(2),
    texttemplate="%{text}",
    textfont={"size": 11},
    hoverongaps=False,
))
fig4.update_layout(
    title="Mapa de Correlación entre Stats Base",
    height=450,
    plot_bgcolor="white",
)
fig4.show()

#### Gráfico 5 — Barras agrupadas: Promedio de stats por tipo

**Objetivo:** Comparar el perfil estadístico de cada tipo  
**Decisión:** Barras agrupadas — permite comparar múltiples variables a la vez  
**Insight:** ¿Qué tipos son ofensivos vs. defensivos?


In [34]:
stats_por_tipo = (
    df.groupby("tipo_primario")[["ataque", "defensa", "velocidad", "hp"]]
    .mean()
    .round(1)
    .reset_index()
    .sort_values("ataque", ascending=False)
    .head(10)  # Top 10 tipos por ataque
)

fig5 = px.bar(
    stats_por_tipo.melt(id_vars="tipo_primario", value_vars=["ataque", "defensa", "velocidad", "hp"]),
    x="tipo_primario",
    y="value",
    color="variable",
    barmode="group",
    title="Promedio de Stats por Tipo (Top 10 tipos por Ataque)",
    labels={"value": "Valor promedio", "tipo_primario": "Tipo", "variable": "Stat"},
    color_discrete_map={
        "ataque": "#FF6B35", "defensa": "#4FC3F7",
        "velocidad": "#66BB6A", "hp": "#EC407A"
    },
)
fig5.update_layout(plot_bgcolor="white", height=450)
fig5.show()

---
## 🏗️ Sección 3: Arquitectura de la Data App

### 3.2.1 — Estructura modular: separar lógica de UI y lógica de datos

Una Data App bien diseñada separa claramente:
- **`data_loader.py`** (o funciones cacheadas): lógica de ingesta y transformación
- **`app.py`**: únicamente lógica de interfaz de usuario

Este es el patrón que sigue nuestro `app.py` completo:


In [35]:
# Pseudocódigo de la arquitectura del app.py
arquitectura = """
app.py
├── IMPORTS (requests, pandas, plotly, streamlit)
├── CONSTANTES (BASE_URL, CACHE_PATH, TIPO_COLORES)
│
├── @st.cache_data → cargar_dataset()     # Una sola carga al inicio
├── @st.cache_data → preparar_datos()     # Transformaciones cacheadas
│
├── configurar_pagina()                   # st.set_page_config
├── sidebar_filtros()                     # st.sidebar — filtros del usuario
│
└── TABS PRINCIPALES
    ├── Tab 1: Resumen General
    │   ├── st.metric × 4 (KPIs)
    │   └── Barras por tipo
    ├── Tab 2: Análisis Comparativo
    │   ├── Scatter ataque vs defensa
    │   └── Box plot HP por tipo
    ├── Tab 3: Explorador de Pokémon
    │   ├── Tabla dinámica filtrable
    │   └── Ficha individual
    └── Tab 4: Correlaciones
        └── Heatmap de stats
"""
print(arquitectura)


app.py
├── IMPORTS (requests, pandas, plotly, streamlit)
├── CONSTANTES (BASE_URL, CACHE_PATH, TIPO_COLORES)
│
├── @st.cache_data → cargar_dataset()     # Una sola carga al inicio
├── @st.cache_data → preparar_datos()     # Transformaciones cacheadas
│
├── configurar_pagina()                   # st.set_page_config
├── sidebar_filtros()                     # st.sidebar — filtros del usuario
│
└── TABS PRINCIPALES
    ├── Tab 1: Resumen General
    │   ├── st.metric × 4 (KPIs)
    │   └── Barras por tipo
    ├── Tab 2: Análisis Comparativo
    │   ├── Scatter ataque vs defensa
    │   └── Box plot HP por tipo
    ├── Tab 3: Explorador de Pokémon
    │   ├── Tabla dinámica filtrable
    │   └── Ficha individual
    └── Tab 4: Correlaciones
        └── Heatmap de stats



---
## ⚡ Sección 4: Patrones de Caché en Streamlit

### 3.2.4 — Por qué el caché es crítico para el rendimiento

Streamlit re-ejecuta TODO el script en cada interacción del usuario.  
Sin caché, la PokeAPI se consulta en cada clic de filtro.


In [36]:
# Demostración del problema SIN caché
import time

def cargar_sin_cache() -> pd.DataFrame:
    """Simula carga sin caché — se ejecuta en cada interacción."""
    time.sleep(0.5)  # Simula latencia de API
    return pd.read_parquet(CACHE_PATH)

# ── Patrón CORRECTO con @st.cache_data ────────────────────────────
# En el app.py real, la función se decora así:
#
# @st.cache_data(ttl=3600)
# def cargar_dataset() -> pd.DataFrame:
#     """Carga datos desde Parquet o PokeAPI."""
#     if CACHE_PATH.exists():
#         return pd.read_parquet(CACHE_PATH)
#     # ... descarga y guarda en Parquet
#
# DIFERENCIA CLAVE:
# - Sin caché: cada filtro → nueva llamada → 2-5 segundos de espera
# - Con caché: primera carga → memoria → todas las interacciones < 50ms

print("Comparación de tiempos:")
print()

inicio = time.time()
for _ in range(3):
    cargar_sin_cache()
sin_cache = time.time() - inicio
print(f"  Sin caché (3 llamadas): {sin_cache:.2f}s")

# Con caché (simulado — en Streamlit real el resultado es instantáneo)
inicio = time.time()
datos_cached = pd.read_parquet(CACHE_PATH)  # Solo una vez en memoria
for _ in range(3):
    _ = datos_cached.copy()  # Uso desde memoria
con_cache = time.time() - inicio
print(f"  Con caché (3 accesos):  {con_cache:.4f}s")
print()
print(f"  Mejora de rendimiento:  {sin_cache/con_cache:.0f}x más rápido")

Comparación de tiempos:

  Sin caché (3 llamadas): 1.51s
  Con caché (3 accesos):  0.0020s

  Mejora de rendimiento:  753x más rápido


In [37]:
# ── Cuándo usar cada decorator ────────────────────────────────────
print("GUÍA DE USO:")
print()
print("@st.cache_data  → Para datos serializables:")
print("  ✓ pd.read_parquet('archivo.parquet')")
print("  ✓ requests.get(API_URL).json()")
print("  ✓ df.groupby(...).mean()")
print("  ✓ Cualquier transformación de pandas")
print()
print("@st.cache_resource → Para objetos NO serializables:")
print("  ✓ Conexiones a base de datos (psycopg2, SQLAlchemy)")
print("  ✓ Modelos de ML cargados (sklearn, tensorflow)")
print("  ✓ Clientes de APIs con estado (boto3, etc.)")
print()
print("PARÁMETROS IMPORTANTES:")
print("  ttl=3600    → Invalida caché cada 1 hora")
print("  ttl=None    → Caché permanente hasta reinicio")
print("  max_entries → Límite de entradas en caché (memoria)")

GUÍA DE USO:

@st.cache_data  → Para datos serializables:
  ✓ pd.read_parquet('archivo.parquet')
  ✓ requests.get(API_URL).json()
  ✓ df.groupby(...).mean()
  ✓ Cualquier transformación de pandas

@st.cache_resource → Para objetos NO serializables:
  ✓ Conexiones a base de datos (psycopg2, SQLAlchemy)
  ✓ Modelos de ML cargados (sklearn, tensorflow)
  ✓ Clientes de APIs con estado (boto3, etc.)

PARÁMETROS IMPORTANTES:
  ttl=3600    → Invalida caché cada 1 hora
  ttl=None    → Caché permanente hasta reinicio
  max_entries → Límite de entradas en caché (memoria)


---
## 🌐 Sección 5: Guía de Despliegue

### 3.2.3 — Streamlit Cloud (recomendado para el proyecto)

**Requisitos previos:**
- Cuenta en GitHub (gratuita)
- Cuenta en Streamlit Cloud: [share.streamlit.io](https://share.streamlit.io)

```
📁 mi-proyecto/
├── app.py                    ← Script principal
├── requirements.txt          ← Dependencias exactas
├── data/
│   └── pokemon_gen1.parquet  ← Caché local (agregar a .gitignore si es grande)
└── .streamlit/
    └── config.toml           ← Configuración visual (opcional)
```

**Pasos:**
1. `git init` → `git add .` → `git commit -m "initial"`
2. Crear repositorio en GitHub → `git push`
3. En Streamlit Cloud: **New app** → seleccionar repo → `app.py`
4. **Deploy** → URL pública lista en ~2 minutos

### 3.2.3 — Render (alternativa multi-framework)

**Start command para Render:**
```bash
streamlit run app.py --server.port=$PORT --server.address=0.0.0.0
```

**render.yaml (opcional):**
```yaml
services:
  - type: web
    name: pokemon-dashboard
    env: python
    buildCommand: pip install -r requirements.txt
    startCommand: streamlit run app.py --server.port=$PORT --server.address=0.0.0.0
```

### 3.2.4 — Checklist pre-despliegue


In [ ]:
checklist = {
    "requirements.txt con versiones exactas": True,
    "Secrets en variables de entorno (no en código)": True,
    "@st.cache_data en funciones de carga": True,
    "Manejo de errores con try/except": True,
    "st.spinner en operaciones lentas": True,
    "Mensajes de feedback al usuario": True,
    "Código modular con type hints": True,
    "README.md con instrucciones de uso": True,
    "Testeado localmente con: streamlit run app.py": True,
}

print("CHECKLIST PRE-DESPLIEGUE:")
print()
for item, ok in checklist.items():
    estado = "✅" if ok else "❌"
    print(f"  {estado}  {item}")
print()
print(">>> Todos los items listos — listo para desplegar 🚀")

---
## ✅ Resumen

En este notebook cubrimos el ciclo completo del Tema 6:

1. **Ingesta** desde PokeAPI con caché Parquet y manejo de errores
2. **Exploración** y feature engineering orientado al dashboard
3. **Visualización estratégica**: 5 gráficos, cada uno justificado por el objetivo analítico
4. **Arquitectura** modular del `app.py` con separación de responsabilidades
5. **Caché** con `@st.cache_data` para rendimiento óptimo
6. **Despliegue** en Streamlit Cloud y Render con checklist pre-producción

El `app.py` completo está disponible como script independiente listo para subir a GitHub y desplegar.
